In [6]:
!git clone https://github.com/Sisaltilshan/NLP_Group_42.git
%cd NLP_Group_42
!git checkout -b feature/sisal-naivebayes-lstm


fatal: destination path 'NLP_Group_42' already exists and is not an empty directory.
/content/NLP_Group_42
Switched to a new branch 'feature/sisal-naivebayes-lstm'


In [15]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


In [16]:
from google.colab import files
uploaded = files.upload()

Saving spam_dataset.csv to spam_dataset (1).csv


In [17]:
df = pd.read_csv('spam_dataset.csv').sample(4000,random_state=1).reset_index(drop=True)
df.head()

,text,label
0,"Yep, by the pretty sculpture",ham
1,"Yes, princess. Are you going to make me moan?",ham
2,Welp apparently he retired,ham
3,Havent.,ham
4,I forgot 2 ask ü all smth.. There's a card on ...,ham


In [19]:
def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'<.*?>', ' ', t)
    t = re.sub(r'http\S+', ' ', t)
    t = re.sub(r'[^a-z\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean'] = df['text'].apply(clean_text)
df[['text', 'clean']].head()

,text,clean
0,"Yep, by the pretty sculpture",yep by the pretty sculpture
1,"Yes, princess. Are you going to make me moan?",yes princess are you going to make me moan
2,Welp apparently he retired,welp apparently he retired
3,Havent.,havent
4,I forgot 2 ask ü all smth.. There's a card on ...,i forgot ask all smth there s a card on da pre...


In [20]:
X_train, X_test, y_train, y_test = train_test_split(df['clean'],
df['label'], test_size=0.2, random_state=1)


In [21]:
vec = TfidfVectorizer(max_features=3000)
X_train_v = vec.fit_transform(X_train)
X_test_v = vec.transform(X_test)
nb = MultinomialNB()
nb.fit(X_train_v, y_train)
pred = nb.predict(X_test_v)
print("Naive Bayes Accuracy:", accuracy_score(y_test, pred))
print("Naive Bayes F1 Score:", f1_score(y_test, pred,
pos_label='spam'))

Naive Bayes Accuracy: 0.9625
Naive Bayes F1 Score: 0.8598130841121495


In [23]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
tok = Tokenizer(num_words=3000)
tok.fit_on_texts(X_train)
X_train_seq = pad_sequences(tok.texts_to_sequences(X_train),
maxlen=80)
X_test_seq = pad_sequences(tok.texts_to_sequences(X_test),
maxlen=80)
y_train_n = (y_train == 'spam').astype(int)
y_test_n = (y_test == 'spam').astype(int)
model = Sequential([
Embedding(3000, 32, input_length=80),
LSTM(32),
Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=
['accuracy'])
model.fit(X_train_seq, y_train_n, epochs=2, batch_size=32,
validation_split=0.1)


Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


90/90 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.8944 - loss: 0.3051 - val_accuracy: 0.9656 - val_loss: 0.1335
Epoch 2/2
90/90 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9781 - loss: 0.0942 - val_accuracy: 0.9937 - val_loss: 0.0478
